# RAG evaluation on German tenancy law

Primary research for *Evaluating the Effectiveness of Retrieval-Augmented Generation (RAG) in Domain-Specific AI Chatbots*.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. It's free and makes the reranker roughly 10x faster. CPU also works if the GPU is unavailable.

Run the cells top to bottom. Cell 4 is the one that must not be skipped.

## 1. Install

In [ ]:
!pip install -q rank_bm25 sentence-transformers pandas matplotlib scipy krippendorff
print('done')

## 2. Upload the project

Run this cell, click **Choose Files**, pick `rag-mietrecht.zip`.

In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()
name = list(up.keys())[0]
with zipfile.ZipFile(name) as z:
    z.extractall('/content')
os.chdir('/content/rag-mietrecht')
print('working dir:', os.getcwd())
print(sorted(os.listdir('data/corpus')))

## 3. API key

Left sidebar → key icon (**Secrets**) → **Add new secret**, with *Notebook access* on.

- Free route: name it `GOOGLE_API_KEY`, key from `aistudio.google.com`.
- Paid route: name it `ANTHROPIC_API_KEY`, key from `console.anthropic.com`.

Never paste a key into a cell — this notebook goes in your appendix.

In [ ]:
import os
from google.colab import userdata

for name in ('GOOGLE_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[name] = userdata.get(name)
        print(f'{name}: loaded')
    except Exception:
        print(f'{name}: not set (fine if you are using the other provider)')

## 4. Confirm the real embedding model loaded

**Do not skip this.** `DenseRetriever` falls back to TF-IDF if `sentence-transformers` fails, and it does so silently. Numbers produced by the fallback are not reportable. This cell fails loudly instead.

In [ ]:
import sys, json
sys.path.insert(0, 'src')
from ingest import build
from retrievers import DenseRetriever

chunks = [c.to_dict() for c in build('data/corpus', 'structural', 900, 150)]
dense = DenseRetriever(chunks)

assert dense.backend == 'sentence-transformers', (
    f'FALLBACK ACTIVE ({dense.backend}) — fix the install before running anything'
)
print(f'OK: {dense.model_name} over {len(chunks)} chunks')

## 5. Retrieval only

No model calls, so no cost. This gives you Recall@k, MRR and nDCG for all four conditions — the quantitative backbone of Chapter 4.1. Two to three minutes.

In [ ]:
!python src/run_experiment.py --dry-run --gold data/gold/gold_v1.jsonl

## 6. Full run with the generator

400 model calls (4 conditions × 100 questions).

`--generator gemini` is free but rate-limited per minute, so expect 30–60 minutes; the retry loop handles the throttling.
`--generator anthropic` costs a few euros and takes 15–25 minutes.

Run the 18-question smoke test first.

In [ ]:
# smoke test first — 18 questions


In [ ]:
# the real run


## 7. Ablations

Retrieval-only, so these are free. Each becomes a table or figure in 4.1.

In [ ]:
# chunking strategy: does splitting on § boundaries beat blind windowing?
for strategy in ['structural', 'fixed']:
    print(f'\n===== chunking = {strategy} =====')
    !python src/run_experiment.py --dry-run --gold data/gold/gold_v1.jsonl --chunking {strategy}

In [ ]:
# chunk size
for size in [512, 900, 1400]:
    print(f'\n===== chunk_size = {size} =====')
    !python src/run_experiment.py --dry-run --gold data/gold/gold_v1.jsonl --chunk-size {size}

In [ ]:
# retrieval depth k
for k in [3, 5, 10]:
    print(f'\n===== k = {k} =====')
    !python src/run_experiment.py --dry-run --gold data/gold/gold_v1.jsonl --k {k}

## 8. Download everything

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/results', 'zip', 'results')
files.download('/content/results.zip')

## 9. Fusion weight sweep

Free. Traces retrieval quality from pure BM25 (0.0) to pure dense (1.0) so you can show *where* fusion stops helping, not just that it doesn't.

In [ ]:
!python src/analyse.py sweep

## 10. Failure analysis

The raw material for Chapter 4.2. Picks the best condition automatically, then breaks the misses down by difficulty and theme and prints each one with what was retrieved instead.

In [ ]:
import glob, os
latest = max(glob.glob('results/run_*.jsonl'), key=os.path.getmtime)
print('analysing', latest, '\n')
!python src/analyse.py failures {latest}

## 11. One-shot report

Everything in one block: setup check, condition table, fusion sweep, miss breakdown.
Run cells 1, 2, 4, 5 first, then this. Copy the whole output.

Contains no answer text and nothing key-shaped, so it is safe to paste.

In [ ]:
!python src/report.py

## 12. Verify the gold set

Start with the failures. Each block shows the question in German and English, the cited provision with an English gloss, the actual statutory text, and what was retrieved instead.

You are checking the label, not the answer: *does this provision govern this topic?*

In [ ]:
import glob, os
latest = max(glob.glob('results/run_*.jsonl'), key=os.path.getmtime)
!python src/verify.py misses {latest}

Then the remaining items:

In [ ]:
!python src/verify.py sheet

List only the QIDs you judged **wrong**. Everything else is marked verified.

In [ ]:
!python src/verify.py mark --wrong Q999

# replace Q999 with your wrong ones, e.g.:
#   !python src/verify.py mark --wrong Q021 Q044
# none wrong? then just:
#   !python src/verify.py mark --wrong

Rerun, then re-report. The setup block should now say `gold verified : 100/100`.

In [ ]:
!python src/run_experiment.py --dry-run --gold data/gold/gold_v1.jsonl
!python src/report.py

---
**Colab disconnects after roughly 90 minutes idle and everything in `/content` is lost.** Download after every run, or mount Drive at the start and write results there instead.